# Agent memory systems on GPU: cross-architecture latency profile

The same workload (LoCoMo conversation replay + QA) is run on five memory architectures:
**RAG baseline**, **mem0 (pgvector)**, **mem0 (Qdrant server)**, **Graphiti/Zep (Neo4j)** and **Letta (Postgres)**.
The local production stack is **vLLM** for the chat LLM and the embedding model, plus native Postgres, Qdrant and Neo4j servers.

**How to run:** Runtime → Change runtime type → GPU. An A100 is recommended; an L4 works for the mid-range profile. Then run the cells in order.
Every run is copied to Google Drive as soon as it finishes. After a disconnect, rerun cells 1–4 and then the suite cell you were on; completed runs are skipped.

Approximate cost on an A100-40G (about 5.4 CU/h, a third-party measurement):

| Step | Time | CU |
|---|---|---|
| Setup | ~0.3 h | ~2 |
| Smoke | ~0.2 h | ~1 |
| e1 | ~1.5 h | ~8 |
| e2 | ~1.5 h | ~8 |
| e3 | ~2.5 h | ~14 |
| e4 | ~1 h | ~6 |

The full A100 pass is about 40 CU. Repeating e1 and e3 on an L4 adds about 15 CU.

In [ ]:
#@title 1. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc; free -g | head -2; df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Code (set GH_TOKEN in Colab Secrets if the repo is private)
import os, subprocess
BRANCH = "claude/gifted-maxwell-td8cnz"
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/envymoon/Agent_Mem_Sys.git"
if not os.path.exists("/content/Agent_Mem_Sys"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/Agent_Mem_Sys"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Agent_Mem_Sys", "pull", "-q"], check=True)
%cd /content/Agent_Mem_Sys
!git log --oneline -1

In [ ]:
#@title 3. Pick the model profile for this GPU, install everything (~15 min, idempotent)
import os, subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
name, mem = gpu.split(",")[0], int(gpu.split(",")[1])
os.environ.update({"LLM_MODEL": "Qwen/Qwen3-8B", "LLM_QUANT": "", "LLM_GPU_UTIL": "0.80", "LLM_CTX": "32768",
                   "EMB_MODEL": "Qwen/Qwen3-Embedding-0.6B", "EMB_GPU_UTIL": "0.08",
                   "LOCOMO": "/content/locomo/data/locomo10.json", "SVC_HOME": "/opt/svc", "SVC_DATA": "/opt/svc/data"})
if "L4" in name:                      # 24 GB Ada: FP8 weights (mid-range consumer-card proxy)
    os.environ.update({"LLM_QUANT": "fp8"})
elif "T4" in name:                    # 16 GB Turing: debugging only
    os.environ.update({"LLM_MODEL": "Qwen/Qwen3-4B-Instruct-2507", "LLM_CTX": "16384", "VLLM_EXTRA": "--dtype half",
                       "EMB_EXTRA": "--dtype half"})
if mem >= 70000:                      # A100-80G / H100: also run the 14B in e4
    os.environ["E4_BIG"] = "Qwen/Qwen3-14B"
os.environ["HW"] = name.replace(" ", "_")
os.environ["DEST"] = f"/content/drive/MyDrive/memprof_results/{os.environ['HW']}"
print(name, mem, "MiB ->", os.environ["LLM_MODEL"], os.environ["LLM_QUANT"] or "bf16", "| results ->", os.environ["DEST"])
!bash scripts/colab_setup.sh 2>&1 | tail -20

In [ ]:
#@title 4. Start services + vLLM (LLM + embedder) + proxy
!scripts/services.sh start all
!scripts/start_vllm.sh all
!curl -s localhost:8100/v1/models | head -c 300; echo
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
#@title 5. Smoke test: every system, 3 turns (~10 min). Check that each run prints a JSON line.
!scripts/run_gpu_suite.sh smoke

In [ ]:
#@title 6. e1: main profile, synchronous write path (100 turns + 50 QA per system)
!scripts/run_gpu_suite.sh e1

In [ ]:
#@title 7. e2: foreground/background interference (async memorize, with and without think time)
!scripts/run_gpu_suite.sh e2

In [ ]:
#@title 8. e3: KV placement (prefix cache off; VRAM KV budget sweep with/without host-RAM KV offload)
!scripts/run_gpu_suite.sh e3

In [ ]:
#@title 9. (optional) e4: model size (Qwen3-4B, plus Qwen3-14B on 80 GB GPUs)
!scripts/run_gpu_suite.sh e4

In [ ]:
#@title 10. Summary tables + archive to Drive
import glob, os
runs = sorted(d for d in glob.glob(f"results/gpu/{os.environ['HW']}/*/") if os.path.exists(d + "meta.json"))
!.venv/bin/python analysis/multi_breakdown.py {" ".join(runs)} | tee results/gpu/{os.environ['HW']}/summary.md
!cp results/gpu/{os.environ['HW']}/summary.md "$DEST/"
!cd results/gpu && tar czf "$DEST/../memprof_{os.environ['HW']}.tgz" {os.environ['HW']} && ls -la "$DEST/.."
print("Upload memprof_<GPU>.tgz to the Claude session, or push it with the next cell.")

In [ ]:
#@title 11. (optional) push results to a separate branch (needs GH_TOKEN with write access)
import os, subprocess
hw = os.environ["HW"]
cmds = f'''
set -e
git config user.email "colab@memprof" && git config user.name "memprof-colab"
git checkout -B results/{hw.lower()}
git add -f results/gpu/{hw}
git commit -qm "GPU results: {hw}" || true
git push -q -u origin results/{hw.lower()}
'''
print(subprocess.run(["bash","-c",cmds], capture_output=True, text=True))